# Notebook 13 — XGBoost + Optuna + Ensemble LightGBM

## Objectif
Améliorer le RMSE=4.07 (notebook 12) via :
1. **XGBoost** avec Optuna — souvent complémentaire à LightGBM sur les mêmes données
2. **Ensemble LightGBM + XGBoost** — moyenne pondérée des prédictions OOF
3. **LightGBM corrigé** — fold 3 n'avait pas convergé (3000 iter), on augmente le budget

## Référence
- LightGBM Optuna (NB 12) : RMSE=4.071, MAE=3.003

## Fichiers utilisés
- `data/X_train.csv`, `data/y_train.csv`, `data/X_test.csv`

## Notebook suivant
`14_ensemble_l2.ipynb` — validation sans fuite et lissage de niveau 2.

## 1. Imports

In [1]:
import json
import os
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
import skore
import xgboost as xgb
from lightgbm import LGBMRegressor
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")
print(f"LightGBM {lgb.__version__}  |  XGBoost {xgb.__version__}")

LightGBM 4.7.0  |  XGBoost 3.4.1


## 2. Connexion Skore Hub

In [2]:
config = json.loads(Path("../.skore").read_text())
os.environ["SKORE_HUB_API_KEY"] = config["api_key"]
skore.login()
hub_project = skore.Project(
    "bobathon-esilv",
    mode="hub",
    workspace=config["workspace"],
)
print(f"Connecté : workspace={hub_project.workspace}")

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                        Successfully logged in, using API key.                        │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Connecté : workspace=BERNIE MADOFF


## 3. Feature engineering (identique NB 12)

In [3]:
CAT = ["cohort", "gene", "rater_id"]


def _slope(x, y):
    m = ~(np.isnan(x) | np.isnan(y))
    if m.sum() < 2 or np.ptp(x[m]) < 1e-9:
        return np.nan, np.nan, m.sum()
    xs, ys = x[m], y[m]
    xm, ym = xs.mean(), ys.mean()
    b = ((xs - xm) * (ys - ym)).sum() / ((xs - xm) ** 2).sum()
    return b, ym - b * xm, m.sum()


def _patient_feats(g, col, pre):
    age, v, n = g["age"].values, g[col].values, len(g)
    out = {}
    b, a, k = _slope(age, v)
    out[f"{pre}_slope"] = np.full(n, b)
    out[f"{pre}_fit"]   = a + b * age if not np.isnan(b) else np.full(n, np.nan)
    out[f"{pre}_mean"]  = np.full(n, np.nanmean(v) if k else np.nan)
    out[f"{pre}_std"]   = np.full(n, np.nanstd(v) if k > 1 else np.nan)
    out[f"{pre}_first"] = np.full(n, v[~np.isnan(v)][0] if k else np.nan)
    out[f"{pre}_last"]  = np.full(n, v[~np.isnan(v)][-1] if k else np.nan)
    out[f"{pre}_cnt"]   = np.full(n, k)
    loo = np.full(n, np.nan)
    for i in range(n):
        mk = np.ones(n, bool); mk[i] = False
        bi, ai, _ = _slope(age[mk], v[mk])
        if not np.isnan(bi):
            loo[i] = ai + bi * age[i]
    out[f"{pre}_loo"] = loo
    m_v = ~np.isnan(v)
    quad = np.full(n, np.nan)
    if m_v.sum() >= 4:
        c = np.polyfit(age[m_v] - age.mean(), v[m_v], 2)
        quad = np.polyval(c, age - age.mean())
    out[f"{pre}_quad"] = quad
    return pd.DataFrame(out, index=g.index)


def build(df, target_enc=None):
    d = df.sort_values(["patient_id", "age", "Index"]).copy()
    gb = d.groupby("patient_id")
    d["dx_missing"]  = d["age_at_diagnosis"].isna().astype(int)
    d["dur"]         = d["age"] - d["age_at_diagnosis"]
    d["dur_first"]   = gb["dur"].transform("min")
    d["age_first"]   = gb["age"].transform("min")
    d["t_rel"]       = d["age"] - d["age_first"]
    d["n_vis"]       = gb["age"].transform("size")
    d["vis_idx"]     = gb.cumcount()
    d["vis_frac"]    = d["vis_idx"] / (d["n_vis"] - 1).clip(lower=1)
    d["gap_prev"]    = gb["age"].diff()
    d["gap_next"]    = -gb["age"].diff(-1)
    d["on_off_diff"]  = d["off"] - d["on"]
    d["on_off_ratio"] = d["on"] / d["off"].replace(0, np.nan)
    d["ledd_per_dur"] = d["ledd"] / d["dur"].clip(lower=0.5)
    d["has_on"]       = d["on"].notna().astype(int)
    d["has_off"]      = d["off"].notna().astype(int)
    d["has_ledd"]     = d["ledd"].notna().astype(int)
    d["tsi_off_dev"]  = d["time_since_intake_off"] - 12
    d["on_x_ledd"]    = d["on"] * d["ledd"]
    d["off_x_ledd"]   = d["off"] * d["ledd"]
    d["age_sq"]       = d["age"] ** 2
    d["dur_sq"]       = d["dur"] ** 2
    for c in ["on", "off", "ledd", "time_since_intake_off", "time_since_intake_on"]:
        for k in (1, 2):
            d[f"{c}_lag{k}"]  = gb[c].shift(k)
            d[f"{c}_lead{k}"] = gb[c].shift(-k)
    d["off_nb_mean"]    = d[["off_lag1", "off_lead1"]].mean(axis=1)
    d["on_nb_mean"]     = d[["on_lag1",  "on_lead1"]].mean(axis=1)
    d["off_delta_prev"] = d["off"] - d["off_lag1"]
    d["off_delta_next"] = d["off_lead1"] - d["off"]
    parts = [
        pd.concat([_patient_feats(g, c, c) for c in ["on", "off", "ledd"]], axis=1)
        for _, g in d.groupby("patient_id", sort=False)
    ]
    d = d.join(pd.concat(parts))
    d["pseudo_off_from_on"] = d["on"] * (d["off_mean"] / d["on_mean"])
    d["traj_blend"]         = d[["off_loo", "pseudo_off_from_on"]].mean(axis=1)
    d["p_age_dx"]           = d.groupby("patient_id")["age_at_diagnosis"].transform("first")
    d["off_loo_ratio"]      = d["off_loo"] / d["off"].replace(0, np.nan)
    d["off_loo_resid"]      = d["off"] - d["off_loo"]
    if target_enc is not None:
        d["rater_target_mean"] = d["rater_id"].map(target_enc).fillna(target_enc.get("__global__", 37.0))
    for c in CAT:
        d[c] = d[c].astype("category")
    return d.sort_values("Index")


def feature_cols(d):
    return [c for c in d.columns if c not in {"Index", "patient_id", "target"}]


print("Feature engineering défini")

Feature engineering défini


## 4. Chargement des données

In [4]:
DATA_DIR = Path("../data")
X_raw      = pd.read_csv(DATA_DIR / "X_train.csv")
y_raw      = pd.read_csv(DATA_DIR / "y_train.csv")
X_test_raw = pd.read_csv(DATA_DIR / "X_test.csv")

merged_full = X_raw.merge(y_raw, on="Index")
rater_enc = merged_full.groupby("rater_id")["target"].mean().to_dict()
rater_enc["__global__"] = merged_full["target"].mean()

d = build(X_raw, target_enc=rater_enc).merge(y_raw, on="Index").sort_values("Index").reset_index(drop=True)

# Pour XGBoost : encoder les catégorielles en codes numériques
CAT_COLS_IDX = [c for c in feature_cols(d) if str(d[c].dtype) == "category"]
d_xgb = d.copy()
for c in CAT_COLS_IDX:
    d_xgb[c] = d_xgb[c].cat.codes.astype("float32").replace(-1, np.nan)

cols     = feature_cols(d)       # pour LightGBM (catégorielles natives)
cols_xgb = feature_cols(d_xgb)   # pour XGBoost (numériques)

print(f"Nb features : {len(cols)}  |  Nb patients : {d['patient_id'].nunique()}")

Nb features : 89  |  Nb patients : 5576


## 5. LightGBM — meilleurs paramètres NB12 + budget augmenté

Le fold 3 du NB12 n'avait pas convergé à 3000 iterations. On passe à `num_boost_round=5000` avec early stopping à 150.

In [5]:
# Meilleurs paramètres trouvés par Optuna dans NB12
LGBM_PARAMS = dict(
    objective="regression",
    verbose=-1,
    num_threads=4,
    bagging_freq=1,
    learning_rate=0.01439,
    num_leaves=74,
    min_child_samples=16,
    feature_fraction=0.5146,
    bagging_fraction=0.6351,
    lambda_l2=0.3732,
    cat_smooth=42,
)

oof_lgbm = np.zeros(len(d))
models_lgbm = []
rmse_lgbm_folds = []

for fold, (tr, va) in enumerate(GroupKFold(n_splits=5).split(d, groups=d["patient_id"])):
    m = lgb.train(
        LGBM_PARAMS,
        lgb.Dataset(d.loc[tr, cols], d.loc[tr, "target"]),
        num_boost_round=5000,
        valid_sets=[lgb.Dataset(d.loc[va, cols], d.loc[va, "target"])],
        callbacks=[lgb.early_stopping(150, verbose=False)],
    )
    oof_lgbm[va] = m.predict(d.loc[va, cols], num_iteration=m.best_iteration)
    models_lgbm.append(m)
    fold_rmse = np.sqrt(((oof_lgbm[va] - d.loc[va, "target"].values) ** 2).mean())
    rmse_lgbm_folds.append(fold_rmse)
    print(f"  LGBM Fold {fold+1} — iter={m.best_iteration:4d}  RMSE={fold_rmse:.4f}")

rmse_lgbm = np.sqrt(((oof_lgbm - d["target"].values) ** 2).mean())
mae_lgbm  = np.abs(oof_lgbm - d["target"].values).mean()
print(f"\nLightGBM OOF — RMSE={rmse_lgbm:.4f}  MAE={mae_lgbm:.4f}")

  LGBM Fold 1 — iter=1800  RMSE=4.1319
  LGBM Fold 2 — iter=2944  RMSE=4.1346
  LGBM Fold 3 — iter=1590  RMSE=4.0650
  LGBM Fold 4 — iter=2578  RMSE=3.9710
  LGBM Fold 5 — iter=3267  RMSE=4.0433

LightGBM OOF — RMSE=4.0696  MAE=3.0024


## 6. XGBoost — Optuna tuning

In [6]:
def objective_xgb(trial):
    params = dict(
        objective="reg:squarederror",
        tree_method="hist",          # rapide, gère les NaN nativement
        n_jobs=4,
        random_state=42,
        verbosity=0,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        min_child_weight=trial.suggest_int("min_child_weight", 5, 50),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
        colsample_bylevel=trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
    )
    oof = np.zeros(len(d))
    for tr, va in GroupKFold(n_splits=3).split(d_xgb, groups=d["patient_id"]):
        model = XGBRegressor(
            **params,
            n_estimators=3000,
            early_stopping_rounds=50,
            eval_metric="rmse",
        )
        model.fit(
            d_xgb.loc[tr, cols_xgb], d_xgb.loc[tr, "target"],
            eval_set=[(d_xgb.loc[va, cols_xgb], d_xgb.loc[va, "target"])],
            verbose=False,
        )
        oof[va] = model.predict(d_xgb.loc[va, cols_xgb])
    return np.sqrt(((oof - d["target"].values) ** 2).mean())


study_xgb = optuna.create_study(direction="minimize",
                                 sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

best_xgb = study_xgb.best_params
print(f"\nMeilleur RMSE XGBoost Optuna (3-fold) : {study_xgb.best_value:.4f}")
print("Meilleurs hyperparamètres XGBoost :")
for k, v in best_xgb.items():
    print(f"  {k}: {v}")

  0%|          | 0/50 [00:00<?, ?it/s]


Meilleur RMSE XGBoost Optuna (3-fold) : 4.1648
Meilleurs hyperparamètres XGBoost :
  learning_rate: 0.010127963257331486
  max_depth: 7
  min_child_weight: 37
  subsample: 0.8645035840204937
  colsample_bytree: 0.8627622080115674
  colsample_bylevel: 0.44442679104045424
  reg_alpha: 0.02715581955282941
  reg_lambda: 0.0029072088906598446
  gamma: 4.315517129377968


## 7. XGBoost — entraînement final 5-fold

In [7]:
oof_xgb = np.zeros(len(d))
models_xgb = []
rmse_xgb_folds = []

for fold, (tr, va) in enumerate(GroupKFold(n_splits=5).split(d_xgb, groups=d["patient_id"])):
    model = XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=4,
        random_state=42,
        verbosity=0,
        n_estimators=5000,
        early_stopping_rounds=150,
        eval_metric="rmse",
        **best_xgb,
    )
    model.fit(
        d_xgb.loc[tr, cols_xgb], d_xgb.loc[tr, "target"],
        eval_set=[(d_xgb.loc[va, cols_xgb], d_xgb.loc[va, "target"])],
        verbose=False,
    )
    oof_xgb[va] = model.predict(d_xgb.loc[va, cols_xgb])
    models_xgb.append(model)

    fold_rmse = np.sqrt(((oof_xgb[va] - d.loc[va, "target"].values) ** 2).mean())
    rmse_xgb_folds.append(fold_rmse)
    print(f"  XGB Fold {fold+1} — iter={model.best_iteration:4d}  RMSE={fold_rmse:.4f}")

rmse_xgb = np.sqrt(((oof_xgb - d["target"].values) ** 2).mean())
mae_xgb  = np.abs(oof_xgb - d["target"].values).mean()
print(f"\nXGBoost OOF — RMSE={rmse_xgb:.4f}  MAE={mae_xgb:.4f}")

  XGB Fold 1 — iter=3877  RMSE=4.1777
  XGB Fold 2 — iter=4526  RMSE=4.1695
  XGB Fold 3 — iter=4973  RMSE=4.0646
  XGB Fold 4 — iter=3798  RMSE=3.9863
  XGB Fold 5 — iter=3466  RMSE=4.0746

XGBoost OOF — RMSE=4.0952  MAE=3.0320


## 8. Ensemble LightGBM + XGBoost

On teste différentes pondérations et on garde celle qui minimise le RMSE OOF.

In [8]:
best_w, best_rmse_ens = 0.5, np.inf
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * oof_lgbm + (1 - w) * oof_xgb
    r = np.sqrt(((blend - d["target"].values) ** 2).mean())
    if r < best_rmse_ens:
        best_rmse_ens, best_w = r, w

oof_ensemble = best_w * oof_lgbm + (1 - best_w) * oof_xgb
rmse_ens = np.sqrt(((oof_ensemble - d["target"].values) ** 2).mean())
mae_ens  = np.abs(oof_ensemble - d["target"].values).mean()

print(f"Poids optimaux : LGBM={best_w:.2f}  XGB={1-best_w:.2f}")
print(f"RMSE LightGBM seul : {rmse_lgbm:.4f}")
print(f"RMSE XGBoost seul  : {rmse_xgb:.4f}")
print(f"RMSE Ensemble      : {rmse_ens:.4f}  MAE={mae_ens:.4f}")
print(f"Gain vs LGBM seul  : {rmse_lgbm - rmse_ens:+.4f}")

Poids optimaux : LGBM=0.65  XGB=0.35
RMSE LightGBM seul : 4.0696
RMSE XGBoost seul  : 4.0952
RMSE Ensemble      : 4.0601  MAE=2.9975
Gain vs LGBM seul  : +0.0095


## 9. Prédictions sur le jeu de test

In [9]:
dt = build(X_test_raw, target_enc=rater_enc).sort_values("Index").reset_index(drop=True)
for c in CAT:
    dt[c] = pd.Categorical(dt[c].astype(str), categories=d[c].cat.categories)

# XGBoost test : encoder catégorielles
dt_xgb = dt.copy()
for c in CAT_COLS_IDX:
    dt_xgb[c] = dt_xgb[c].cat.codes.astype("float32").replace(-1, np.nan)

# Prédictions LightGBM (ensemble 5 folds)
preds_lgbm = np.mean(
    [m.predict(dt[cols], num_iteration=m.best_iteration) for m in models_lgbm],
    axis=0,
)

# Prédictions XGBoost (ensemble 5 folds)
preds_xgb = np.mean(
    [m.predict(dt_xgb[cols_xgb]) for m in models_xgb],
    axis=0,
)

# Blend avec les poids optimaux
final_preds = np.clip(
    best_w * preds_lgbm + (1 - best_w) * preds_xgb,
    0, 100,
)

print(f"Min: {final_preds.min():.2f}  Max: {final_preds.max():.2f}  Mean: {final_preds.mean():.2f}")

Min: 0.00  Max: 90.74  Mean: 36.73


## 10. Soumission

In [10]:
submission = X_test_raw[["Index"]].copy()
submission["target"] = final_preds

assert list(submission.columns) == ["Index", "target"]
assert len(submission) == len(X_test_raw)
assert submission["Index"].equals(X_test_raw["Index"])
assert submission["target"].notna().all()

Path("../submissions").mkdir(exist_ok=True)
submission.to_csv("../submissions/xgb_ensemble.csv", index=False)
print(f"Soumission enregistrée : submissions/xgb_ensemble.csv ({len(submission)} lignes)")
display(submission.head())

Soumission enregistrée : submissions/xgb_ensemble.csv (11013 lignes)


,Index,target
0,66,22.610732
1,67,30.913708
2,68,28.958851
3,69,31.956419
4,70,35.314948


## 11. Envoi Skore Hub

In [11]:
# Rapport LightGBM via LGBMRegressor (sklearn wrapper)
n_est_lgbm = int(np.mean([m.best_iteration for m in models_lgbm]))
lgbm_skore = LGBMRegressor(
    n_estimators=n_est_lgbm, learning_rate=LGBM_PARAMS["learning_rate"],
    num_leaves=LGBM_PARAMS["num_leaves"], min_child_samples=LGBM_PARAMS["min_child_samples"],
    feature_fraction=LGBM_PARAMS["feature_fraction"], bagging_fraction=LGBM_PARAMS["bagging_fraction"],
    bagging_freq=1, reg_lambda=LGBM_PARAMS["lambda_l2"], cat_smooth=LGBM_PARAMS["cat_smooth"],
    random_state=0, verbose=-1,
)
cv_sk = list(GroupKFold(n_splits=5).split(d[cols], d["target"], groups=d["patient_id"]))
report_lgbm = skore.evaluate(lgbm_skore, d[cols], d["target"], splitter=cv_sk)
hub_project.put("11_lgbm_final", report_lgbm)
print(f"LGBM skore RMSE : {float(report_lgbm.metrics.rmse().iloc[:, 0].values.mean()):.4f}")

# Rapport XGBoost via XGBRegressor
n_est_xgb = int(np.mean([m.best_iteration for m in models_xgb]))
xgb_skore = XGBRegressor(
    objective="reg:squarederror", tree_method="hist", n_jobs=4,
    random_state=42, verbosity=0, n_estimators=n_est_xgb, **best_xgb,
)
cv_sk_xgb = list(GroupKFold(n_splits=5).split(d_xgb[cols_xgb], d["target"], groups=d["patient_id"]))
report_xgb = skore.evaluate(xgb_skore, d_xgb[cols_xgb], d["target"], splitter=cv_sk_xgb)
hub_project.put("12_xgb_final", report_xgb)
print(f"XGB  skore RMSE : {float(report_xgb.metrics.rmse().iloc[:, 0].values.mean()):.4f}")
print("Rapports envoyés : 11_lgbm_final, 12_xgb_final")

Output()

Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/cross-validations/70430

Output()

LGBM skore RMSE : 4.0729


Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/cross-validations/70452

XGB  skore RMSE : 4.0956
Rapports envoyés : 11_lgbm_final, 12_xgb_final


## 12. Récapitulatif final

In [12]:
recap = pd.DataFrame([
    {"Modèle": "tabular_pipeline (skrub)",        "RMSE OOF": 7.53,       "MAE OOF": 5.93},
    {"Modèle": "LightGBM base (NB11)",             "RMSE OOF": 4.1412,     "MAE OOF": 3.0760},
    {"Modèle": "LightGBM + Optuna (NB12)",         "RMSE OOF": 4.0711,     "MAE OOF": 3.0027},
    {"Modèle": "LightGBM Optuna + budget (NB13)",  "RMSE OOF": rmse_lgbm,  "MAE OOF": mae_lgbm},
    {"Modèle": "XGBoost Optuna (NB13)",            "RMSE OOF": rmse_xgb,   "MAE OOF": mae_xgb},
    {"Modèle": f"Ensemble LGBM({best_w:.2f})+XGB({1-best_w:.2f})",
                                                    "RMSE OOF": rmse_ens,   "MAE OOF": mae_ens},
])
display(recap.round(4))
print(f"\nFichier soumission : submissions/xgb_ensemble.csv")

,Modèle,RMSE OOF,MAE OOF
0,tabular_pipeline (skrub),7.5300,5.9300
1,LightGBM base (NB11),4.1412,3.0760
2,LightGBM + Optuna (NB12),4.0711,3.0027
3,LightGBM Optuna + budget (NB13),4.0696,3.0024
4,XGBoost Optuna (NB13),4.0952,3.0320
5,Ensemble LGBM(0.65)+XGB(0.35),4.0601,2.9975



Fichier soumission : submissions/xgb_ensemble.csv
